# TuneMap NL-to-Cypher — Eval Driver

Runs the full eval pipeline on a cloud GPU (Colab A100 / Vast.ai RTX 3090+).

**Steps:**
1. Repo setup — clone or locate the repo root
2. Install dependencies
3. Set credentials (HF + Neo4j AuraDB)
4. Regenerate `eval.jsonl`
5. Run `translation_eval.py` → `translation_report.json`
6. Run `execution_eval.py` → `execution_report.json` *(requires AuraDB)*

**VRAM requirement**: Qwen3.5-9B bf16 needs ~18 GB. Free Colab T4 (16 GB) will OOM — use Colab Pro (A100) or Vast.ai.

## 1. Repo setup

In [ ]:
import os
from pathlib import Path

REPO_NAME = "AppleMusciKG"
REPO_URL  = "https://github.com/pariidanDKE/AppleMusciKG.git"
BRANCH    = "cyper_finetune"

# Walk up from CWD to find the repo root (handles running from any subdirectory)
def find_repo_root(marker="training"):
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).is_dir():
            return parent
    return None

root = find_repo_root()

if root is None:
    print(f"Repo not found locally — cloning {REPO_URL} ...")
    os.system(f"git clone --branch {BRANCH} {REPO_URL}")
    root = Path.cwd() / REPO_NAME

os.chdir(root)
print(f"Working directory: {Path.cwd()}")
assert (Path.cwd() / "training").is_dir(), "training/ not found — check repo root"

## 2. Install dependencies

In [ ]:
# torch must be pre-installed or installed separately first.
# On Colab / Vast.ai PyTorch images, torch+CUDA is already present.
!pip install -r training/requirements.txt

## 3. Credentials

Set these before running any further cells.  
On Colab: use *Secrets* (left sidebar 🔑) and read via `google.colab.userdata`.  
On Vast.ai: paste values directly or load from a `.env` you `scp`'d in.

In [ ]:
import os

# --- Option A: Colab Secrets ---
# from google.colab import userdata
# os.environ["HF_TOKEN"]          = userdata.get("HF_TOKEN")
# os.environ["NEO4J_URI"]         = userdata.get("NEO4J_URI")   # neo4j+s://xxxx.databases.neo4j.io
# os.environ["NEO4J_USER"]        = userdata.get("NEO4J_USER")
# os.environ["NEO4J_PASSWORD"]    = userdata.get("NEO4J_PASSWORD")

# --- Option B: paste directly (Vast.ai / local) ---
os.environ["HF_TOKEN"]          = "hf_..."          # HuggingFace token (read access)
os.environ["NEO4J_URI"]         = "neo4j+s://..."   # AuraDB connection URI
os.environ["NEO4J_USER"]        = "neo4j"
os.environ["NEO4J_PASSWORD"]    = "..."

# Adapter location — local checkpoint or HF repo ID
CHECKPOINT = "danp27/qwen3.5-9b-nl2cypher-lora"

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("NEO4J_URI  set:", bool(os.environ.get("NEO4J_URI")))
print("Checkpoint:", CHECKPOINT)

## 4. Regenerate `eval.jsonl`

Downloads `neo4j/text2cypher-2024v1` and writes `training/data/eval.jsonl`.  
TuneMap gate will be **CLOSED** on cloud (no `cypher_dataset_validated.jsonl`) — external rows only.

In [ ]:
!python training/prepare_data.py

## 5. Translation eval (GLEU — no DB required)

Runs adapter + baseline pass over all 4,833 eval rows.  
Expected runtime: ~5–6 h on a single GPU.  
Output: `training/outputs/translation_report.json`

In [ ]:
!python training/translation_eval.py --checkpoint {CHECKPOINT}

In [ ]:
# Quick summary of translation results
import json
report = json.loads(open("training/outputs/translation_report.json").read())
print(f"Adapter mean GLEU:  {report['mean_gleu']:.4f}")
print(f"Baseline mean GLEU: {report['baseline']['mean_gleu']:.4f}")
print(f"Rows evaluated:     {report['n_translation']}")
print("\n--- 3 worst failures ---")
for f in report["failures"][:3]:
    print(f"  GLEU {f['gleu']:.3f} | Q: {f['question'][:80]}")
    print(f"          Ref: {f['reference_cypher'][:100]}")
    print(f"          Pred: {f['predicted_cypher'][:100]}")

## 6. Execution eval (TuneMap AuraDB — requires NEO4J_* credentials)

Runs adapter + baseline pass over ~120 `source="tunemap"` eval rows.  
Requires a live AuraDB instance.  
Output: `training/outputs/execution_report.json`

In [ ]:
!python training/execution_eval.py --checkpoint {CHECKPOINT}

In [ ]:
# Quick summary of execution results
import json
report = json.loads(open("training/outputs/execution_report.json").read())
print(f"TuneMap rows:              {report['n_tunemap']}")
print(f"Adapter syntax valid:      {report['tunemap_syntax_valid_pct']:.1%}")
print(f"Adapter exec exact match:  {report['tunemap_exec_exact_match_pct']:.1%}")
print(f"Baseline syntax valid:     {report['baseline']['tunemap_syntax_valid_pct']:.1%}")
print(f"Baseline exec exact match: {report['baseline']['tunemap_exec_exact_match_pct']:.1%}")

## 7. Download results

Pull both report files back to your local machine.

In [ ]:
# Colab: download directly
# from google.colab import files
# files.download("training/outputs/translation_report.json")
# files.download("training/outputs/execution_report.json")

# Vast.ai: copy from your local terminal
# scp -P <port> root@<ip>:/root/AppleMusciKG/training/outputs/translation_report.json .
# scp -P <port> root@<ip>:/root/AppleMusciKG/training/outputs/execution_report.json .

print("See comments above for download instructions.")